# V4 2.5D INT8 inference on the FPGA board ARM CPU

Runs the INT8 quantized model on the board ARM CPU. Evaluates
INT8 software quantization on the ARM host processor to match the
FP32 board reference protocol. Existing NPY test volumes are reused.


In [ ]:

try:
    import onnxruntime
except ImportError:
    if wheel_matches:
        import onnxruntime
        print('[Setup] Successfully installed onnxruntime from offline wheel!')
    else:

import threading
import time
import json
import os
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pynq
import torch
import torch.nn.functional as F
from scipy.ndimage import gaussian_filter, map_coordinates

PIPELINE_VERSION = "v4-canonical-letterbox-int8-arm"
WINDOW_RADIUS = 3
N_STACK = 8
INPUT_HEIGHT, INPUT_WIDTH = 112, 96
INPUT_CHANNELS = 16
CANONICAL_VOLUME_SHAPE = (96, 112, 96)
PAD_VALUE = -1.0
SEG_LABELS = [2, 3, 4, 7, 8, 10, 11, 12, 13, 14, 15, 16, 17, 18, 26, 28]
CPU_EXPORT_HELPER = "./vxm_2p5d_export.py"
WEIGHTS_PATH = "./2p5d_dense_pt_v4_canonical_best.pth"
OUTPUT_PATH = "int8_arm_board_results_v4.json"
PAIR_LIMIT = None
LATENCY_REPETITIONS = 3
MINIMUM_POWER_WINDOW_S = 10.0
POWER_SAMPLE_INTERVAL_S = 0.10
IDLE_CALIBRATION_S = 20.0

def resolve_file(default_path, extra_candidates=None):
    candidates = [Path(default_path)]
    candidates.extend(Path(item) for item in (extra_candidates or []))
    for candidate in candidates:
        if candidate.exists():
            return str(candidate)
    return str(candidates[0])

for required in (CPU_EXPORT_HELPER, WEIGHTS_PATH):
    if not Path(required).exists():
        raise FileNotFoundError(
            f"Missing {required}; extract the complete bundle here."
        )
print("Target: FPGA-board ARM CPU")
print("Precision: INT8 (software quantized)")
print("Power scope: monitored PSINTFP + INT rails")


## Validated V4 preprocessing and runtime

Derived from the current FPGA inference notebook.

In [ ]:
import torch
import torch.nn.functional as F


def normalize_volume_contract(volume):
    arr = volume.astype(np.float32)
    arr_min = float(arr.min())
    arr_max = float(arr.max())
    if arr_max <= arr_min:
        return np.zeros_like(arr, dtype=np.float32)
    if arr_min >= -1.001 and arr_max <= 1.001:
        if arr_min >= -1e-4 and arr_max <= 1.0001:
            return (2.0 * arr - 1.0).astype(np.float32)
        return arr
    return (2.0 * (arr - arr_min) / (arr_max - arr_min) - 1.0).astype(np.float32)


def resample_volume_v4(volume, is_segmentation=False):
    """Resample a raw [D,H,W] volume to V4's shared canonical grid."""
    tensor = torch.from_numpy(np.ascontiguousarray(volume.astype(np.float32)))[None, None]
    if is_segmentation:
        result = F.interpolate(tensor, size=CANONICAL_VOLUME_SHAPE, mode='nearest')
        return np.ascontiguousarray(result[0, 0].numpy().astype(np.int16))
    result = F.interpolate(tensor, size=CANONICAL_VOLUME_SHAPE, mode='trilinear', align_corners=False)
    return np.ascontiguousarray(result[0, 0].numpy().astype(np.float32))


def extract_v4_stack(volume, orientation, z):
    """Extract V4's native seven-slice stack from a canonical [D,H,W] volume."""
    if orientation == 'axial':
        stack = volume[z - WINDOW_RADIUS:z + WINDOW_RADIUS + 1]
    elif orientation == 'coronal':
        stack = volume[:, z - WINDOW_RADIUS:z + WINDOW_RADIUS + 1, :].transpose(1, 0, 2)
    elif orientation == 'sagittal':
        stack = volume[:, :, z - WINDOW_RADIUS:z + WINDOW_RADIUS + 1].transpose(2, 0, 1)
    else:
        raise ValueError(f'Unknown orientation: {orientation}')
    if stack.shape[0] != N_STACK - 1:
        raise RuntimeError(f'Expected seven source slices, got {stack.shape}')
    return np.ascontiguousarray(stack)


def _canonicalize_orientation(array, orientation):
    return np.ascontiguousarray(np.swapaxes(array, -2, -1)) if orientation == 'sagittal' else np.ascontiguousarray(array)


def letterbox_stack_v4(stack, orientation):
    """Convert a native V4 stack to [8,112,96] without aspect-ratio distortion."""
    if stack.shape[0] == N_STACK - 1:
        stack = np.concatenate([stack, stack[-1:]], axis=0)
    if stack.shape[0] != N_STACK:
        raise ValueError(f'Expected {N_STACK} stack slices, got {stack.shape}')
    canonical = _canonicalize_orientation(stack, orientation)
    out = np.full((N_STACK, INPUT_HEIGHT, INPUT_WIDTH), PAD_VALUE, dtype=np.float32)
    if canonical.shape[1:] == (INPUT_HEIGHT, INPUT_WIDTH):
        out[...] = canonical
    elif orientation == 'coronal' and canonical.shape[1:] == (96, 96):
        out[:, 8:104, :] = canonical
    else:
        raise ValueError(f'Unexpected {orientation} plane shape: {canonical.shape[1:]}')
    return np.ascontiguousarray(out)


def letterbox_plane_v4(plane, orientation, is_segmentation=False):
    canonical = _canonicalize_orientation(plane, orientation)
    pad_value = 0 if is_segmentation else PAD_VALUE
    out = np.full((INPUT_HEIGHT, INPUT_WIDTH), pad_value, dtype=np.int16 if is_segmentation else np.float32)
    if canonical.shape == (INPUT_HEIGHT, INPUT_WIDTH):
        out[...] = canonical
    elif orientation == 'coronal' and canonical.shape == (96, 96):
        out[8:104, :] = canonical
    else:
        raise ValueError(f'Unexpected {orientation} plane shape: {canonical.shape}')
    return np.ascontiguousarray(out)


def canvas_flow_to_native_v4(flow, orientation):
    """Undo V4 canvas geometry; return native [N,2,H,W] flow for lifting."""
    if orientation == 'axial':
        return flow
    if orientation == 'coronal':
        return np.ascontiguousarray(flow[:, :, 8:104, :])
    native = np.empty((flow.shape[0], 2, 96, 112), dtype=np.float32)
    native[:, 0] = flow[:, 1].transpose(0, 2, 1)
    native[:, 1] = flow[:, 0].transpose(0, 2, 1)
    return native


def apply_flow_2d(image, flow, interpolation=cv2.INTER_LINEAR):
    h, w = image.shape
    y_coords, x_coords = np.mgrid[0:h, 0:w].astype(np.float32)
    return cv2.remap(image.astype(np.float32), x_coords + flow[0], y_coords + flow[1], interpolation, borderMode=cv2.BORDER_CONSTANT)


def _quiver_vis_gain(flow, target_p95=2.0, max_gain=20.0):
    magnitude = np.sqrt(flow[0] ** 2 + flow[1] ** 2)
    p95 = float(np.percentile(magnitude, 95))
    return (1.0 if p95 <= 1e-8 else float(np.clip(target_p95 / p95, 1.0, max_gain))), p95


def plot_quiver(ax, flow, background=None, step=4, auto_gain=True, target_p95=2.0, max_gain=20.0, display_flip_y=False):
    h, w = flow.shape[1:]
    gain, p95 = _quiver_vis_gain(flow, target_p95, max_gain) if auto_gain else (1.0, 0.0)
    y, x = np.mgrid[0:h:step, 0:w:step]
    fx, fy = flow[0, ::step, ::step] * gain, flow[1, ::step, ::step] * gain
    if display_flip_y:
        fy = -fy
    ax.imshow(np.asarray(background) if background is not None else np.zeros((h, w)), cmap='gray', origin='upper')
    ax.quiver(x, y, fx, fy, color='red', angles='xy', scale_units='xy', scale=1, headwidth=3, headlength=4, alpha=0.8)
    ax.set_xlim(0, w - 1); ax.set_ylim(h - 1, 0)
    return gain, p95

In [ ]:
SEG_LABELS = [2, 3, 4, 7, 8, 10, 11, 12, 13, 14, 15, 16, 17, 18, 26, 28]


def compute_dice_per_label(seg_a, seg_b, labels=SEG_LABELS):
    scores = []
    for label in labels:
        a, b = (seg_a == label), (seg_b == label)
        union = float(a.sum() + b.sum())
        scores.append(1.0 if union == 0.0 else float(2.0 * (a * b).sum() / union))
    return np.asarray(scores, dtype=np.float32)


def mutual_information_np(a, b, bins=64, clip_range=(-1.0, 1.0)):
    hist, _, _ = np.histogram2d(a.ravel(), b.ravel(), bins=bins, range=[clip_range, clip_range])
    pxy = hist / max(hist.sum(), 1.0); px, py = pxy.sum(axis=1, keepdims=True), pxy.sum(axis=0, keepdims=True)
    nz = pxy > 0
    return float((pxy[nz] * np.log(pxy[nz] / (px @ py)[nz])).sum())


def structural_similarity_np(a, b, data_range=2.0, sigma=1.5, truncate=3.5, eps=1e-8):
    mu_a, mu_b = gaussian_filter(a, sigma=sigma, mode='reflect', truncate=truncate), gaussian_filter(b, sigma=sigma, mode='reflect', truncate=truncate)
    c1, c2 = (0.01 * data_range) ** 2, (0.03 * data_range) ** 2
    var_a = np.maximum(gaussian_filter(a * a, sigma=sigma, mode='reflect', truncate=truncate) - mu_a * mu_a, 0.0)
    var_b = np.maximum(gaussian_filter(b * b, sigma=sigma, mode='reflect', truncate=truncate) - mu_b * mu_b, 0.0)
    cov = gaussian_filter(a * b, sigma=sigma, mode='reflect', truncate=truncate) - mu_a * mu_b
    return float(((2.0 * mu_a * mu_b + c1) * (2.0 * cov + c2) / np.maximum((mu_a * mu_a + mu_b * mu_b + c1) * (var_a + var_b + c2), eps)).mean())


def warp_volume_3d_numpy(volume, flow_xyz, mode='linear'):
    d, h, w = volume.shape
    zz, yy, xx = np.meshgrid(np.arange(d), np.arange(h), np.arange(w), indexing='ij')
    coords = np.stack([zz + flow_xyz[2], yy + flow_xyz[1], xx + flow_xyz[0]], axis=0)
    return map_coordinates(volume.astype(np.float32), coords, order=1 if mode == 'linear' else 0, mode='nearest')


def resize_component_3d(component, target_shape):
    tensor = torch.from_numpy(component[None, None].astype(np.float32))
    return F.interpolate(tensor, size=target_shape, mode='trilinear', align_corners=False)[0, 0].numpy().astype(np.float32)


def canonical_field_to_raw(field_xyz, raw_shape):
    scale = (raw_shape[2] / 96.0, raw_shape[1] / 112.0, raw_shape[0] / 96.0)
    return np.stack([resize_component_3d(field_xyz[channel], raw_shape) * factor for channel, factor in enumerate(scale)]).astype(np.float32)


def lift_axial_v4(flow_stack, canonical_shape=CANONICAL_VOLUME_SHAPE):
    out = np.zeros((3, *canonical_shape), dtype=np.float32)
    for index, z in enumerate(range(WINDOW_RADIUS, canonical_shape[0] - WINDOW_RADIUS)):
        out[0, z], out[1, z] = flow_stack[index, 0], flow_stack[index, 1]
    return out


def lift_coronal_v4(flow_stack, canonical_shape=CANONICAL_VOLUME_SHAPE):
    out = np.zeros((3, *canonical_shape), dtype=np.float32)
    for index, y in enumerate(range(WINDOW_RADIUS, canonical_shape[1] - WINDOW_RADIUS)):
        out[0, :, y, :], out[2, :, y, :] = flow_stack[index, 0], flow_stack[index, 1]
    return out


def lift_sagittal_v4(flow_stack, canonical_shape=CANONICAL_VOLUME_SHAPE):
    out = np.zeros((3, *canonical_shape), dtype=np.float32)
    for index, x in enumerate(range(WINDOW_RADIUS, canonical_shape[2] - WINDOW_RADIUS)):
        out[1, :, :, x], out[2, :, :, x] = flow_stack[index, 0], flow_stack[index, 1]
    return out


def fuse_fields(axial, coronal, sagittal):
    fused = np.zeros_like(axial)
    fused[0], fused[1], fused[2] = 0.5 * (axial[0] + coronal[0]), 0.5 * (axial[1] + sagittal[1]), 0.5 * (coronal[2] + sagittal[2])
    return fused


def smooth_field_3d(field, sigma=0.75):
    return np.stack([gaussian_filter(component, sigma=sigma).astype(np.float32) for component in field])


def _label_centroid_dhw(segmentation, label):
    coordinates = np.argwhere(segmentation == label)
    return None if coordinates.size == 0 else coordinates.mean(axis=0).astype(np.float32)


def label_centroid_tre_metrics(moving_seg, warped_seg, fixed_seg, spacing_dhw_mm=(1.0, 1.0, 1.0)):
    spacing = np.asarray(spacing_dhw_mm, dtype=np.float32)
    before, after, per_label, missing = [], [], {}, []
    for label in SEG_LABELS:
        moving_center, fixed_center = _label_centroid_dhw(moving_seg, label), _label_centroid_dhw(fixed_seg, label)
        if moving_center is None or fixed_center is None: continue
        before.append(float(np.linalg.norm((moving_center - fixed_center) * spacing)))
        warped_center = _label_centroid_dhw(warped_seg, label)
        if warped_center is None:
            missing.append(int(label)); continue
        error = float(np.linalg.norm((warped_center - fixed_center) * spacing))
        after.append(error); per_label[str(int(label))] = error
    return {'tre_before_mm': float(np.mean(before)) if before else None, 'tre_mm': float(np.mean(after)) if after else None, 'tre_median_mm': float(np.median(after)) if after else None, 'tre_label_count': len(after), 'tre_missing_warped_labels': missing, 'tre_per_label_mm': per_label}


def summarize_registration(moving_vol, fixed_vol, moving_seg, fixed_seg, flow, warped_vol, warped_seg):
    dice_before, dice_after = compute_dice_per_label(moving_seg, fixed_seg), compute_dice_per_label(warped_seg, fixed_seg)
    metrics = {'mi_before': mutual_information_np(moving_vol, fixed_vol), 'mi_after': mutual_information_np(warped_vol, fixed_vol), 'ssim_deformed_fixed': structural_similarity_np(warped_vol, fixed_vol), 'ssim_deformed_moving': structural_similarity_np(warped_vol, moving_vol), 'dice_before': float(dice_before.mean()), 'dice_after': float(dice_after.mean()), 'flow_min': float(flow.min()), 'flow_max': float(flow.max()), 'flow_mean': float(flow.mean()), 'flow_std': float(flow.std())}
    metrics.update(label_centroid_tre_metrics(moving_seg, warped_seg, fixed_seg))
    return metrics


MODEL_CPU = None

def run_cpu_inference(moving_stack, fixed_stack, weights_path):
    global MODEL_CPU
    if MODEL_CPU is None:
        import sys
        helper_dir = str(Path(CPU_EXPORT_HELPER).resolve().parent)
        if helper_dir not in sys.path: sys.path.insert(0, helper_dir)
        from vxm_2p5d_export import load_quantized_model_for_export as load_model_for_export
        print(f'Loading V4 PyTorch INT8 quantized CPU model from {weights_path}...')
        MODEL_CPU = load_model_for_export(weights_path).to('cpu').eval()
    input_data = np.concatenate([moving_stack, fixed_stack], axis=0).transpose(1, 2, 0)[None].astype(np.float32)
    start = time.perf_counter()
    with torch.no_grad(): flow = MODEL_CPU(torch.from_numpy(input_data)).squeeze(0).numpy()
    return flow.astype(np.float32), time.perf_counter() - start


def infer_v4_orientation(moving_c, fixed_c, orientation, device, weights_path=None):
    axis = {'axial': 0, 'coronal': 1, 'sagittal': 2}[orientation]
    flows, times = [], []
    for z in range(WINDOW_RADIUS, moving_c.shape[axis] - WINDOW_RADIUS):
        moving_stack = letterbox_stack_v4(extract_v4_stack(moving_c, orientation, z), orientation)
        fixed_stack = letterbox_stack_v4(extract_v4_stack(fixed_c, orientation, z), orientation)
        flow, elapsed = run_cpu_inference(moving_stack, fixed_stack, weights_path)
        flows.append(flow); times.append(elapsed)
    return canvas_flow_to_native_v4(np.stack(flows).astype(np.float32), orientation), times


class PowerMonitor:
    def __init__(self, device_name, sample_interval_s=0.10, idle_dpu_w=0.0, idle_cpu_w=0.0):
        self.device_name = device_name
        self.sample_interval_s = sample_interval_s
        self.idle_dpu_w = float(idle_dpu_w)
        self.idle_cpu_w = float(idle_cpu_w)
        self.gpu_samples_w = []
        self.cpu_samples_w = []
        self.memory_samples_mb = []
        self._stop = False
        self._thread = None
        self._start_rss_mb = None
        self._end_rss_mb = None
        self._start_wall = None
        self._end_wall = None
        self.rails = pynq.get_rails()

    @staticmethod
    def _read_rss_mb():
        try:
            with open('/proc/self/status', 'r') as f:
                for line in f:
                    if line.startswith('VmRSS:'):
                        return float(line.split()[1]) / 1024.0
        except Exception:
            pass
        return 0.0

    def _sample_power_and_memory(self):
        while not self._stop:
            if 'INT' in self.rails and self.rails['INT'].power:
                self.gpu_samples_w.append(self.rails['INT'].power.value)
            if 'PSINTFP' in self.rails and self.rails['PSINTFP'].power:
                self.cpu_samples_w.append(self.rails['PSINTFP'].power.value)
            self.memory_samples_mb.append(self._read_rss_mb())
            time.sleep(self.sample_interval_s)

    def __enter__(self):
        self._start_rss_mb = self._read_rss_mb()
        self.memory_samples_mb.append(self._start_rss_mb)
        self._start_wall = time.perf_counter()
        self._stop = False
        self._thread = threading.Thread(target=self._sample_power_and_memory, daemon=True)
        self._thread.start()
        return self

    def __exit__(self, exc_type, exc, tb):
        self._end_wall = time.perf_counter()
        self._end_rss_mb = self._read_rss_mb()
        self.memory_samples_mb.append(self._end_rss_mb)
        self._stop = True
        if self._thread is not None:
            self._thread.join(timeout=1.0)
        return False

    def result(self):
        wall_time_s = 0.0
        if self._start_wall is not None and self._end_wall is not None:
            wall_time_s = max(float(self._end_wall - self._start_wall), 0.0)

        gpu_power_mean_w = None
        gpu_power_peak_w = None
        gpu_energy_j = None
        gpu_dynamic_energy_j = None
        if self.gpu_samples_w:
            gpu_power_mean_w = float(np.mean(self.gpu_samples_w))
            gpu_power_peak_w = float(np.max(self.gpu_samples_w))
            gpu_energy_j = float(gpu_power_mean_w * wall_time_s)
            gpu_dynamic_energy_j = float(max(gpu_energy_j - (self.idle_dpu_w * wall_time_s), 0.0))

        cpu_power_mean_w = None
        cpu_energy_j = None
        cpu_dynamic_energy_j = None
        if self.cpu_samples_w:
            cpu_power_mean_w = float(np.mean(self.cpu_samples_w))
            cpu_energy_j = float(cpu_power_mean_w * wall_time_s)
            cpu_dynamic_energy_j = float(max(cpu_energy_j - (self.idle_cpu_w * wall_time_s), 0.0))

        energy_parts = [v for v in [cpu_energy_j, gpu_energy_j] if v is not None]
        energy_j = float(sum(energy_parts)) if energy_parts else None
        
        dynamic_energy_parts = [v for v in [cpu_dynamic_energy_j, gpu_dynamic_energy_j] if v is not None]
        dynamic_energy_j = float(sum(dynamic_energy_parts)) if dynamic_energy_parts else None

        power_mean_w = None
        if energy_j is not None and wall_time_s > 0:
            power_mean_w = float(energy_j / wall_time_s)

        process_rss_peak_mb = None
        process_rss_delta_mb = None
        if self.memory_samples_mb:
            process_rss_peak_mb = float(np.max(self.memory_samples_mb))
        if self._start_rss_mb is not None and self._end_rss_mb is not None:
            process_rss_delta_mb = float(self._end_rss_mb - self._start_rss_mb)

        return {
            'power_wall_time_s': wall_time_s,
            'cpu_energy_j': cpu_energy_j,
            'cpu_dynamic_energy_j': cpu_dynamic_energy_j,
            'cpu_power_mean_w': cpu_power_mean_w,
            'gpu_energy_j': gpu_energy_j,
            'gpu_dynamic_energy_j': gpu_dynamic_energy_j,
            'gpu_power_mean_w': gpu_power_mean_w,
            'gpu_power_peak_w': gpu_power_peak_w,
            'gpu_power_samples': int(len(self.gpu_samples_w)),
            'energy_j': energy_j,
            'dynamic_energy_j': dynamic_energy_j,
            'power_mean_w': power_mean_w,
            'process_rss_start_mb': self._start_rss_mb,
            'process_rss_end_mb': self._end_rss_mb,
            'process_rss_peak_mb': process_rss_peak_mb,
            'process_rss_delta_mb': process_rss_delta_mb,
            'process_memory_samples': int(len(self.memory_samples_mb)),
        }

def combine_power_measurements(parts):
    parts = [p for p in parts if p]
    wall_time_s = sum(float(p.get('power_wall_time_s') or 0.0) for p in parts)

    def sum_known(key):
        vals = [p.get(key) for p in parts if p.get(key) is not None]
        return None if not vals else float(sum(vals))

    cpu_energy_j = sum_known('cpu_energy_j')
    cpu_dynamic_energy_j = sum_known('cpu_dynamic_energy_j')
    gpu_energy_j = sum_known('gpu_energy_j')
    gpu_dynamic_energy_j = sum_known('gpu_dynamic_energy_j')
    energy_j = sum_known('energy_j')
    dynamic_energy_j = sum_known('dynamic_energy_j')

    gpu_peak_vals = [p.get('gpu_power_peak_w') for p in parts if p.get('gpu_power_peak_w') is not None]
    gpu_samples = int(sum(int(p.get('gpu_power_samples') or 0) for p in parts))
    rss_peak_vals = [p.get('process_rss_peak_mb') for p in parts if p.get('process_rss_peak_mb') is not None]
    rss_start_vals = [p.get('process_rss_start_mb') for p in parts if p.get('process_rss_start_mb') is not None]
    rss_end_vals = [p.get('process_rss_end_mb') for p in parts if p.get('process_rss_end_mb') is not None]
    memory_samples = int(sum(int(p.get('process_memory_samples') or 0) for p in parts))
    rss_start = None if not rss_start_vals else float(rss_start_vals[0])
    rss_end = None if not rss_end_vals else float(rss_end_vals[-1])

    return {
        'power_wall_time_s': float(wall_time_s),
        'cpu_energy_j': cpu_energy_j,
        'cpu_dynamic_energy_j': cpu_dynamic_energy_j,
        'cpu_power_mean_w': None if cpu_energy_j is None or wall_time_s <= 0 else float(cpu_energy_j / wall_time_s),
        'gpu_energy_j': gpu_energy_j,
        'gpu_dynamic_energy_j': gpu_dynamic_energy_j,
        'gpu_power_mean_w': None if gpu_energy_j is None or wall_time_s <= 0 else float(gpu_energy_j / wall_time_s),
        'gpu_power_peak_w': None if not gpu_peak_vals else float(max(gpu_peak_vals)),
        'gpu_power_samples': gpu_samples,
        'energy_j': energy_j,
        'dynamic_energy_j': dynamic_energy_j,
        'power_mean_w': None if energy_j is None or wall_time_s <= 0 else float(energy_j / wall_time_s),
        'process_rss_start_mb': rss_start,
        'process_rss_end_mb': rss_end,
        'process_rss_peak_mb': None if not rss_peak_vals else float(max(rss_peak_vals)),
        'process_rss_delta_mb': None if rss_start is None or rss_end is None else float(rss_end - rss_start),
        'process_memory_samples': memory_samples,
    }

def attach_power(summary, power):
    for key, value in power.items():
        summary[key] = value
    return summary


In [ ]:
helper_dir = str(Path(CPU_EXPORT_HELPER).resolve().parent)
if helper_dir not in os.sys.path:
    os.sys.path.insert(0, helper_dir)
from vxm_2p5d_export import load_quantized_model_for_export as load_model_for_export

MODEL_CPU = load_model_for_export(WEIGHTS_PATH).to("cpu").eval()
PARAMETER_COUNT = sum(p.numel() for p in MODEL_CPU.parameters())
with torch.inference_mode():
    output = MODEL_CPU(torch.zeros(
        1, INPUT_HEIGHT, INPUT_WIDTH, INPUT_CHANNELS
    ))
assert tuple(output.shape) == (1, 2, INPUT_HEIGHT, INPUT_WIDTH)
print(f"INT8 Model ready: {PARAMETER_COUNT:,} parameters")


## INT8 ARM benchmark

Latency is unmonitored. A separate identical execution records raw
power and energy. Raw power is not idle-subtracted and is not
whole-system wall power.


In [ ]:
from fpga_deployment_timing import PAIRS
from fpga_deployment_power import run_benchmark

selected_pairs = PAIRS if PAIR_LIMIT is None else PAIRS[:PAIR_LIMIT]
results = run_benchmark(
    globals(),
    output_path=OUTPUT_PATH,
    latency_repetitions=LATENCY_REPETITIONS,
    minimum_power_window_s=MINIMUM_POWER_WINDOW_S,
    power_sample_interval_s=POWER_SAMPLE_INTERVAL_S,
    idle_calibration_s=IDLE_CALIBRATION_S,
    devices=("cpu",),
    pairs=selected_pairs,
)
arm = results["models"]["2.5d_fused_arm_cpu"]
throughput = 1000.0 / arm["total_runtime_ms_mean"]
performance_per_watt = throughput / arm["power_mean_w_mean"]
arm["throughput_inferences_per_s"] = throughput
arm["performance_per_watt_inferences_per_s_per_w"] = performance_per_watt
results["execution"] = {
    "target": "FPGA board ARM CPU",
    "precision": "INT8",
    "quantized": True,
    "accelerator": "none",
    "fusion": "mean",
    "parameter_count": PARAMETER_COUNT,
}
Path(OUTPUT_PATH).write_text(json.dumps(results, indent=2), encoding="utf-8")
print(f"Latency: {arm['total_runtime_ms_mean'] / 1000.0:.3f} s")
print(f"Raw mean power: {arm['power_mean_w_mean']:.3f} W")
print(f"Raw energy: {arm['energy_j_per_inference_mean']:.3f} J/inference")
print(f"Performance/W: {performance_per_watt:.5f} inf/s/W")
